# Draft-01: Titanic EDA + Logistic Regression (Kaggle)

Single-notebook baseline with the same Kaggle workflow style as your root draft: package checks, `.env` loading, Kaggle API auth, optional competition download, fallback path resolution, real submission file creation, and optional auto submission.


In [ ]:
import importlib.util
import os
import subprocess
import sys
from datetime import UTC, datetime
from pathlib import Path
from zipfile import ZipFile

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from dotenv import load_dotenv
from IPython.display import display
from kaggle.api.kaggle_api_extended import KaggleApi
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler


def run_cmd(cmd):
    print("Running:", " ".join(cmd))
    subprocess.check_call(cmd)


def ensure_package(module_name, package_name=None):
    pkg = package_name or module_name
    if importlib.util.find_spec(module_name) is not None:
        print(f"{pkg} is already installed")
        return
    run_cmd([sys.executable, "-m", "pip", "install", pkg])


ensure_package("dotenv", "python-dotenv")
ensure_package("kaggle")

pd.set_option("display.max_columns", 200)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (12, 5)


In [ ]:
COMPETITION = "titanic"
RUN_DOWNLOAD = True
RUN_SUBMISSION = True
FORCE_DOWNLOAD = True
RANDOM_STATE = 42
TEST_SIZE = 0.2

ID_COL = "PassengerId"
LABEL_COL = "Survived"
NOTEBOOK_SLUG = "titnic-01-eda-and-modeling"


def resolve_data_paths_fallback():
    kaggle_candidates = [
        Path("/kaggle/input/titanic"),
        Path("/kaggle/input/titanic-competition"),
        Path("/kaggle/input"),
    ]
    local_candidates = [
        Path.cwd(),
        Path.cwd() / "data",
        Path.cwd().parent / "data",
    ]

    candidate_triplets = []
    for base in kaggle_candidates + local_candidates:
        candidate_triplets.extend(
            [
                (base / "train.csv", base / "test.csv", base / "gender_submission.csv"),
                (
                    base / "titanic" / "train.csv",
                    base / "titanic" / "test.csv",
                    base / "titanic" / "gender_submission.csv",
                ),
                (
                    base / "data" / "train.csv",
                    base / "data" / "test.csv",
                    base / "data" / "gender_submission.csv",
                ),
            ]
        )

    for train_path, test_path, sample_path in candidate_triplets:
        if train_path.exists() and test_path.exists() and sample_path.exists():
            return train_path, test_path, sample_path

    raise FileNotFoundError(
        "Could not find Titanic train/test/gender_submission files in Kaggle input or local folders"
    )


def prepare_competition_data(api_client, competition, data_dir, force_download=False):
    data_dir = Path(data_dir)
    data_dir.mkdir(parents=True, exist_ok=True)
    zip_path = data_dir / f"{competition}.zip"
    extract_dir = data_dir / competition

    if force_download or not zip_path.exists():
        files_resp = api_client.competition_list_files(competition)
        try:
            api_client.competition_download_files(
                competition,
                path=str(data_dir),
                force=force_download,
                quiet=False,
            )
            print("Competition zip download complete")
        except Exception as download_error:
            print(
                "Bulk download failed, fallback to per-file download:", download_error
            )
            for f in files_resp.files:
                file_path = data_dir / f.name
                if file_path.exists() and not force_download:
                    continue
                api_client.competition_download_file(
                    competition=competition,
                    file_name=f.name,
                    path=str(data_dir),
                    force=force_download,
                    quiet=False,
                )

    if zip_path.exists():
        if not extract_dir.exists() or not any(extract_dir.iterdir()):
            extract_dir.mkdir(parents=True, exist_ok=True)
            with ZipFile(zip_path, "r") as zf:
                zf.extractall(extract_dir)

    train_path = extract_dir / "train.csv"
    test_path = extract_dir / "test.csv"
    sample_path = extract_dir / "gender_submission.csv"

    if not (train_path.exists() and test_path.exists() and sample_path.exists()):
        fallback_train = data_dir / "train.csv"
        fallback_test = data_dir / "test.csv"
        fallback_sample = data_dir / "gender_submission.csv"
        if (
            fallback_train.exists()
            and fallback_test.exists()
            and fallback_sample.exists()
        ):
            train_path, test_path, sample_path = (
                fallback_train,
                fallback_test,
                fallback_sample,
            )
        else:
            raise FileNotFoundError(
                "Could not resolve Titanic train/test/gender_submission files"
            )

    return train_path, test_path, sample_path, zip_path, extract_dir


In [ ]:
load_dotenv(".env", override=True)
os.environ.pop("KAGGLE_API_TOKEN", None)

api = None
try:
    api = KaggleApi()
    api.authenticate()
    print("Authenticated user:", api.get_config_value("username"))
except Exception as auth_error:
    print("Kaggle API auth not ready:", auth_error)
    print("Falling back to existing /kaggle/input or local data if available")

if RUN_DOWNLOAD:
    if api is None:
        raise RuntimeError("RUN_DOWNLOAD=True but Kaggle API auth is not available")
    BASE_DIR = Path.cwd()
    DATA_DIR = BASE_DIR / "data"
    TRAIN_PATH, TEST_PATH, SAMPLE_PATH, ZIP_PATH, EXTRACT_DIR = (
        prepare_competition_data(
            api_client=api,
            competition=COMPETITION,
            data_dir=DATA_DIR,
            force_download=FORCE_DOWNLOAD,
        )
    )
else:
    TRAIN_PATH, TEST_PATH, SAMPLE_PATH = resolve_data_paths_fallback()
    BASE_DIR = TRAIN_PATH.parent
    ZIP_PATH = None
    EXTRACT_DIR = TRAIN_PATH.parent

OUTPUT_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path.cwd()
OUTPUT_PATH = OUTPUT_DIR / f"submission_{NOTEBOOK_SLUG}.csv"
DIAGNOSTICS_PATH = OUTPUT_DIR / f"diagnostics_{NOTEBOOK_SLUG}.csv"
VALIDATION_PATH = OUTPUT_DIR / f"validation_predictions_{NOTEBOOK_SLUG}.csv"

config_df = pd.DataFrame(
    [
        {
            "competition": COMPETITION,
            "run_download": RUN_DOWNLOAD,
            "run_submission": RUN_SUBMISSION,
            "force_download": FORCE_DOWNLOAD,
            "random_state": RANDOM_STATE,
            "train_path": str(TRAIN_PATH),
            "test_path": str(TEST_PATH),
            "sample_path": str(SAMPLE_PATH),
            "output_file": str(OUTPUT_PATH),
        }
    ]
)
display(config_df)


In [ ]:
train_df = pd.read_csv(TRAIN_PATH)
test_df = pd.read_csv(TEST_PATH)
sample_submission = pd.read_csv(SAMPLE_PATH)

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)
print("Sample submission shape:", sample_submission.shape)
display(train_df.head())
display(sample_submission.head())

missing_df = train_df.isna().sum().rename("missing").to_frame()
missing_df["missing_pct"] = missing_df["missing"] / len(train_df)
display(missing_df[missing_df["missing"] > 0].sort_values("missing", ascending=False))


In [ ]:
survival_by_sex = train_df.groupby("Sex")[LABEL_COL].mean().sort_values(ascending=False)
survival_by_class = train_df.groupby("Pclass")[LABEL_COL].mean().sort_index()

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
axes[0].bar(survival_by_sex.index, survival_by_sex.values)
axes[0].set_title("Survival Rate by Sex")
axes[0].set_ylim(0, 1)

axes[1].bar(survival_by_class.index.astype(str), survival_by_class.values)
axes[1].set_title("Survival Rate by Pclass")
axes[1].set_ylim(0, 1)

train_df["Age"].dropna().hist(bins=30, ax=axes[2], edgecolor="black")
axes[2].set_title("Age Distribution")
axes[2].set_xlabel("Age")

plt.tight_layout()
plt.show()

eda_summary = pd.DataFrame(
    [
        {
            "survival_rate": train_df[LABEL_COL].mean(),
            "female_survival_rate": survival_by_sex.get("female", np.nan),
            "male_survival_rate": survival_by_sex.get("male", np.nan),
            "first_class_survival_rate": survival_by_class.get(1, np.nan),
            "third_class_survival_rate": survival_by_class.get(3, np.nan),
        }
    ]
)
display(eda_summary)


In [ ]:
def add_features(df):
    df = df.copy()
    df["Title"] = (
        df["Name"].str.extract(r" ([A-Za-z]+)\.", expand=False).fillna("Unknown")
    )
    df["Title"] = df["Title"].where(
        df["Title"].isin(["Mr", "Mrs", "Miss", "Master"]), "Rare"
    )
    df["FamilySize"] = df["SibSp"] + df["Parch"] + 1
    df["IsAlone"] = (df["FamilySize"] == 1).astype(int)
    df["CabinDeck"] = df["Cabin"].fillna("U").str[0]
    return df


train_features = add_features(train_df)
test_features = add_features(test_df)

numeric_features = ["Pclass", "Age", "SibSp", "Parch", "Fare", "FamilySize", "IsAlone"]
categorical_features = ["Sex", "Embarked", "Title", "CabinDeck"]
feature_cols = numeric_features + categorical_features

X = train_features[feature_cols]
y = train_features[LABEL_COL]
X_test = test_features[feature_cols]

display(train_features[feature_cols + [LABEL_COL]].head())


In [ ]:
X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y,
)

preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            Pipeline(
                [
                    ("imputer", SimpleImputer(strategy="median")),
                    ("scaler", StandardScaler()),
                ]
            ),
            numeric_features,
        ),
        (
            "cat",
            Pipeline(
                [
                    ("imputer", SimpleImputer(strategy="most_frequent")),
                    ("onehot", OneHotEncoder(handle_unknown="ignore")),
                ]
            ),
            categorical_features,
        ),
    ]
)

pipeline = Pipeline(
    [
        ("preprocessor", preprocessor),
        ("model", LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)),
    ]
)

pipeline.fit(X_train, y_train)
y_val_pred = pipeline.predict(X_val)

val_accuracy = accuracy_score(y_val, y_val_pred)
val_precision = precision_score(y_val, y_val_pred)
val_recall = recall_score(y_val, y_val_pred)
val_f1 = f1_score(y_val, y_val_pred)

metrics_df = pd.DataFrame(
    [
        {
            "validation_accuracy": val_accuracy,
            "validation_precision": val_precision,
            "validation_recall": val_recall,
            "validation_f1": val_f1,
            "train_rows": len(X_train),
            "validation_rows": len(X_val),
        }
    ]
)
display(metrics_df)

print("Classification report:")
print(classification_report(y_val, y_val_pred))


In [ ]:
cm = confusion_matrix(y_val, y_val_pred)

fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=ax)
ax.set_title("Validation Confusion Matrix")
ax.set_xlabel("Predicted")
ax.set_ylabel("Actual")
plt.tight_layout()
plt.show()

validation_df = X_val.copy()
validation_df["actual"] = y_val.values
validation_df["predicted"] = y_val_pred
validation_df.to_csv(VALIDATION_PATH, index=False)
metrics_df.to_csv(DIAGNOSTICS_PATH, index=False)

feature_names = pipeline.named_steps["preprocessor"].get_feature_names_out()
coef_df = pd.DataFrame(
    {
        "feature": feature_names,
        "coefficient": pipeline.named_steps["model"].coef_[0],
    }
).sort_values("coefficient", ascending=False)
display(coef_df.head(15))

print("Diagnostics saved to", DIAGNOSTICS_PATH)
print("Validation predictions saved to", VALIDATION_PATH)


In [ ]:
final_pipeline = Pipeline(
    [
        ("preprocessor", preprocessor),
        ("model", LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)),
    ]
)
final_pipeline.fit(X, y)
test_predictions = final_pipeline.predict(X_test)

submission_df = sample_submission.copy()
submission_df[LABEL_COL] = test_predictions
submission_df.to_csv(OUTPUT_PATH, index=False)

display(submission_df.head(10))
print("Submission file saved to", OUTPUT_PATH)
print("Predicted survival rate:", f"{submission_df[LABEL_COL].mean():.2%}")


In [ ]:
if RUN_SUBMISSION:
    if api is None:
        raise RuntimeError("RUN_SUBMISSION=True but Kaggle API auth is not available")
    submit_message = (
        f"{NOTEBOOK_SLUG} logistic baseline | "
        f"val_acc={val_accuracy:.6f} "
        f"val_f1={val_f1:.6f} "
        f"time={datetime.now(UTC).strftime('%Y-%m-%d %H:%M:%S')}Z"
    )
    response = api.competition_submit(
        file_name=str(OUTPUT_PATH),
        message=submit_message,
        competition=COMPETITION,
        quiet=False,
    )
    print("Submission response:", response)
else:
    print("RUN_SUBMISSION is False. File is ready at", OUTPUT_PATH)


## Notes

- Set `RUN_DOWNLOAD = True` when you want the notebook to fetch competition files through the Kaggle API into a local `data/` folder.
- Keep `RUN_DOWNLOAD = False` inside Kaggle if the competition data is already mounted under `/kaggle/input`.
- Set `RUN_SUBMISSION = True` only when you want the notebook to auto submit the generated CSV.
